In [1]:
import sklearn as sk
import pandas as pd
import numpy as np
import optuna
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetRegressor
from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
import torch
import contextlib
import os
import time
import pickle
import warnings
import random
warnings.filterwarnings("ignore")
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor


c:\Users\tweiz\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Data Fetching**

In [5]:
data = fetch_california_housing()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

y = df["target"]
x = df.drop(columns=["target"])
print(x.head())
print('------------------')
print(y)
print('shape:',df.shape)

#print number of features
print('number of features:',x.shape[1])

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  
------------------
0        4.526
1        3.585
2        3.521
3        3.413
4        3.422
         ...  
20635    0.781
20636    0.771
20637    0.923
20638    0.847
20639    0.894
Name: target, Length: 20640, dtype: float64
shape: (20640, 9)
number of features: 8


In [ ]:
#check for missing values
print('missing values:',df.isnull().sum())
#check for duplicates
print('dup_values:',df.duplicated().sum())


missing values: MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
target        0
dtype: int64
dup_values: 0


**TabNet**
---
Since California Housing contains only numeric features with no missing values, the only preprocessing required for TabNet is StandardScaler normalization. The data is split into 60/20/20 train/validation/test sets, with the scaler fit on the training set only and applied to the validation and test sets accordingly to prevent data leakage.

In [ ]:
# 60/20/20 split
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.4, random_state=0)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

y_train_arr = y_train.values.reshape(-1, 1).astype(np.float32)
y_val_arr = y_val.values.reshape(-1, 1).astype(np.float32)
y_test_arr = y_test.values.reshape(-1, 1).astype(np.float32)

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `n_d` / `n_a` (kept equal): controls the width of the decision step
- `n_steps`: number of sequential attention steps
- `lr`: learning rate

In [ ]:


# hyperparameter tuning on train/val only
def objective(trial):
    n_da = trial.suggest_int("n_da", 8, 32, step=8)
    params = {
        "n_da": n_da,
        "n_steps": trial.suggest_int("n_steps", 3, 10),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    }

    model = TabNetRegressor(
        n_d=params["n_da"],
        n_a=params["n_da"],
        n_steps=params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        optimizer_params=dict(lr=params["lr"]),
        seed=0,
        verbose=0
    )

    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_scaled, y_train_arr,
            eval_set=[(X_val_scaled, y_val_arr)],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )

    preds = model.predict(X_val_scaled)
    return np.sqrt(mean_squared_error(y_val_arr, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-04-07 21:14:50,086] A new study created in memory with name: no-name-76907e69-4c25-473d-9803-abcbde881355
[I 2026-04-07 21:15:00,793] Trial 0 finished with value: 1.956284005353998 and parameters: {'n_da': 32, 'n_steps': 8, 'lr': 0.0001213959247006697}. Best is trial 0 with value: 1.956284005353998.
[I 2026-04-07 21:15:07,075] Trial 1 finished with value: 1.4910185864094083 and parameters: {'n_da': 8, 'n_steps': 10, 'lr': 0.003071648561842517}. Best is trial 1 with value: 1.4910185864094083.
[I 2026-04-07 21:15:16,837] Trial 2 finished with value: 1.690614792657242 and parameters: {'n_da': 24, 'n_steps': 10, 'lr': 0.0003165053123293539}. Best is trial 1 with value: 1.4910185864094083.
[I 2026-04-07 21:15:26,273] Trial 3 finished with value: 1.110358165228142 and parameters: {'n_da': 16, 'n_steps': 5, 'lr': 0.0007133460808157787}. Best is trial 3 with value: 1.110358165228142.
[I 2026-04-07 21:15:41,387] Trial 4 finished with value: 0.6987712856182949 and parameters: {'n_da': 32

Best RMSE: 0.6475474902452167
Best params: {'n_da': 32, 'n_steps': 3, 'lr': 0.005966184271860252}


**Final evaluation over 3 seeds**

In [ ]:
# final evaluation across 3 seeds
best_params = study.best_params
seeds = [1, 2, 3]
final_rmse, final_mae = [], []

for seed in seeds:
    model = TabNetRegressor(
        n_d=best_params["n_da"],
        n_a=best_params["n_da"],
        n_steps=best_params["n_steps"],
        gamma=1.3,
        lambda_sparse=1e-5,
        optimizer_params=dict(lr=best_params["lr"]),
        seed=seed,
        verbose=0
    )
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        model.fit(
            X_train_scaled, y_train_arr,
            eval_set=[(X_val_scaled, y_val_arr)],
            max_epochs=30,
            patience=3,
            batch_size=1024,
            virtual_batch_size=128
        )
    preds = model.predict(X_test_scaled)
    final_rmse.append(np.sqrt(mean_squared_error(y_test_arr, preds)))
    final_mae.append(mean_absolute_error(y_test_arr, preds))

print(f"TabNet RMSE: {np.mean(final_rmse):.4f} ± {np.std(final_rmse):.4f}")
print(f"TabNet MAE:  {np.mean(final_mae):.4f} ± {np.std(final_mae):.4f}")

TabNet RMSE: 0.8344 ± 0.0860
TabNet MAE:  0.5947 ± 0.0661


**Inference Cost**

In [ ]:
start = time.time()
_ = model.predict(X_test_scaled)
elapsed = time.time() - start
print(f"Inference time per sample: {elapsed/len(X_test_scaled)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model)) / 1024:.2f} KB")

Inference time per sample: 0.0113 ms
Model size: 1681.93 KB


**Export results**

In [ ]:
#export obtained results, including feature importance by ranking
results = {
    "Model": ["TabNet"],
    "Dataset": ["California Housing"],
    "Task": ["Regression"],
    "RMSE Mean": [np.mean(final_rmse)],
    "RMSE Std": [np.std(final_rmse)],
    "MAE Mean": [np.mean(final_mae)],
    "MAE Std": [np.std(final_mae)],
    "Inference Time (ms/sample)": [elapsed/len(X_test_scaled)*1000],
    "Model Size (KB)": [len(pickle.dumps(model)) / 1024],
    "Feature Ranking (most to least important)": [str(pd.Series(model.feature_importances_, index=x.columns).sort_values(ascending=False).index.tolist())]
}
results_df = pd.DataFrame(results)
results_df.to_csv("results/results_california_housing.csv", index=False)
print(results_df)

    Model             Dataset        Task  RMSE Mean  RMSE Std  MAE Mean  \
0  TabNet  California Housing  Regression   0.834352  0.085951  0.594673   

    MAE Std  Inference Time (ms/sample)  Model Size (KB)  \
0  0.066078                    0.011277      1681.930664   

           Feature Ranking (most to least important)  
0  ['Latitude', 'MedInc', 'Longitude', 'HouseAge'...  


**FT-Transformer**
---
Since California Housing contains only numeric features with no missing values, the only preprocessing required is StandardScaler normalization. The same 60/20/20 train/validation/test split is reused from TabNet, with the scaler already fit on the training set only.

In [ ]:
# reconstruct dataframes for pytorch-tabular (requires dataframes not arrays)
train_df = pd.DataFrame(X_train_scaled, columns=x.columns)
train_df["target"] = y_train.values
val_df = pd.DataFrame(X_val_scaled, columns=x.columns)
val_df["target"] = y_val.values
test_df = pd.DataFrame(X_test_scaled, columns=x.columns)
test_df["target"] = y_test.values

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:
- `attn_heads`: number of attention heads in each transformer block (4 or 8)
- `num_layers`: number of transformer blocks
- `lr`: learning rate

In [ ]:
def objective_ft(trial):
    attn_heads = trial.suggest_categorical("attn_heads", [4, 8])
    num_layers = trial.suggest_int("num_layers", 2, 6)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    data_config = DataConfig(
        target=["target"],
        continuous_cols=list(x.columns),
        categorical_cols=[]
    )
    trainer_config = TrainerConfig(
        max_epochs=30,
        early_stopping="valid_loss",
        early_stopping_patience=3,
        checkpoints=None,
        load_best=False,
        progress_bar="none",
        trainer_kwargs={"enable_model_summary": False}
    )
    optimizer_config = OptimizerConfig()
    model_config = FTTransformerConfig(
        task="regression",
        num_attn_blocks=num_layers,
        num_heads=attn_heads,
        learning_rate=lr
    )
    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config,
    )
    model_ft.fit(train=train_df, validation=val_df)
    preds = model_ft.predict(val_df)["target_prediction"].values
    rmse = np.sqrt(mean_squared_error(val_df["target"].values, preds))
    return rmse

study_ft = optuna.create_study(direction="minimize")
study_ft.optimize(objective_ft, n_trials=50)
print("Best RMSE:", study_ft.best_value)
print("Best params:", study_ft.best_params)

[I 2026-04-07 21:20:10,925] A new study created in memory with name: no-name-951c0a97-b1f6-4372-af8a-fc7ced33de6f
2026-04-07 21:20:10,933 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-07 21:20:10,949 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-07 21:20:10,950 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-04-07 21:20:10,959 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-07 21:20:10,972 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-07 21:20:10,982 - {pytorch_tab

Best RMSE: 0.6101428217638782
Best params: {'attn_heads': 4, 'num_layers': 2, 'lr': 0.0018112337183407957}


In [ ]:
best_params_ft = study_ft.best_params
seeds = [1, 2, 3]
final_rmse_ft, final_mae_ft = [], []

for seed in seeds:
    # Set PyTorch seeds to affect data shuffling/splitting
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    data_config = DataConfig(
        target=["target"],
        continuous_cols=list(x.columns),
        categorical_cols=[]
    )

    trainer_config = TrainerConfig(
        max_epochs=30,
        early_stopping="valid_loss",
        early_stopping_patience=3,
        checkpoints=None,
        load_best=False,
        progress_bar="none",
        seed=seed,  # Pass seed to TrainerConfig
        trainer_kwargs={
            "enable_model_summary": False
        }
    )

    optimizer_config = OptimizerConfig()

    model_config = FTTransformerConfig(
        task="regression",
        num_attn_blocks=best_params_ft["num_layers"],
        num_heads=best_params_ft["attn_heads"],
        learning_rate=best_params_ft["lr"]
    )

    model_ft = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config
    )

    # Shuffle training data with the current seed to affect batch ordering during training
    train_indices = np.random.permutation(len(train_df))
    train_df_shuffled = train_df.iloc[train_indices].reset_index(drop=True)
    
    model_ft.fit(train=train_df_shuffled, validation=val_df)
    preds = model_ft.predict(test_df)["target_prediction"].values

    rmse = np.sqrt(mean_squared_error(test_df["target"].values, preds))
    mae = mean_absolute_error(test_df["target"].values, preds)

    final_rmse_ft.append(rmse)
    final_mae_ft.append(mae)

    print(f"seed={seed}, rmse={rmse:.10f}, mae={mae:.10f}")

print(final_rmse_ft)
print(final_mae_ft)
print(f"FT-Transformer RMSE: {np.mean(final_rmse_ft):.4f} ± {np.std(final_rmse_ft):.4f}")
print(f"FT-Transformer MAE: {np.mean(final_mae_ft):.4f} ± {np.std(final_mae_ft):.4f}")

2026-04-07 21:37:16,608 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-07 21:37:16,625 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-07 21:37:16,627 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-04-07 21:37:16,638 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-07 21:37:16,650 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-07 21:37:16,662 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
2026-04-07 21:37:35,122 - {pytorch_tabular.tabular_model:690} - 

seed=1, rmse=0.6726689669, mae=0.4492489522


2026-04-07 21:37:45,407 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-04-07 21:37:45,581 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-04-07 21:37:45,596 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-04-07 21:37:45,598 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-04-07 21:37:45,609 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-04-07 21:37:45,620 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-04-07 21:37:45,632 - {pytorch_tabular.tabular_m

seed=2, rmse=0.7027362930, mae=0.4657249148


2026-04-07 21:37:59,712 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed


seed=3, rmse=0.6579494413, mae=0.4558521344
[np.float64(0.6726689669112011), np.float64(0.7027362929909294), np.float64(0.6579494413316221)]
[0.44924895216264, 0.46572491476592165, 0.4558521344086967]
FT-Transformer RMSE: 0.6778 ± 0.0186
FT-Transformer MAE: 0.4569 ± 0.0068


NOTE: 

Since pytorch_tabular internally calls seed_everything(42) regardless of the seed parameter passed to TrainerConfig, we implement an alternative approach: shuffling the training data order with different seeds before training. Even though weight initialization remains locked to seed 42, different batch orderings during training produce results in varying final model weights and predictions.

**Inference cost**

In [ ]:
# inference cost
start = time.time()
_ = model_ft.predict(test_df)
elapsed_ft = time.time() - start
print(f"Inference time per sample: {elapsed_ft/len(test_df)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model_ft)) / 1024:.2f} KB")

Inference time per sample: 0.0402 ms
Model size: 1455.07 KB


**Feature importance**
Unlike Tabnet, FT-Transformer has no built-in feature ranking system. Hence, we rank features based on permutation importance. This works by measuring how much the RMSE increases when each feature's values are randomly shuffled — a larger increase indicates a more important feature.

In [ ]:
#need get feature ranking manually since no built-in method for FT-Transformer
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator

class ModelWrapper(BaseEstimator):
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        df = pd.DataFrame(X, columns=x.columns)
        preds = model_ft.predict(df)["target_prediction"].to_numpy().ravel()
        return preds

wrapper = ModelWrapper()


perm = permutation_importance(
    estimator=wrapper,
    X=X_test_scaled,
    y=y_test.values,
    n_repeats=5,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

print(perm.importances_mean)

ft_feature_ranking = (
    pd.Series(perm.importances_mean, index=x.columns)
    .sort_values(ascending=False)
    .index
    .tolist()
)

print(ft_feature_ranking)

[ 5.90841198e-01  4.27518150e-02  9.42331470e-03  7.47909177e-03
 -3.08606341e-04  2.21952839e-02  3.79454348e-01  3.47523664e-01]
['MedInc', 'Latitude', 'Longitude', 'HouseAge', 'AveOccup', 'AveRooms', 'AveBedrms', 'Population']


**Export Results**

In [ ]:
#export obtained results, including feature importance by ranking
new_row = {
    "Model": ["FT-Transformer"],
    "Dataset": ["California Housing"],
    "Task": ["Regression"],
    "RMSE Mean": [np.mean(final_rmse_ft)],
    "RMSE Std": [np.std(final_rmse_ft)],
    "MAE Mean": [np.mean(final_mae_ft)],
    "MAE Std": [np.std(final_mae_ft)],
    "Inference Time (ms/sample)": [elapsed_ft/len(test_df)*1000],
    "Model Size (KB)": [len(pickle.dumps(model_ft)) / 1024],
    "Feature Ranking (most to least important)": [ft_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_california_housing.csv", index=False)
print(results_df)

            Model             Dataset        Task  RMSE Mean  RMSE Std  \
0          TabNet  California Housing  Regression   0.834352  0.085951   
1  FT-Transformer  California Housing  Regression   0.677785  0.018639   

   MAE Mean   MAE Std  Inference Time (ms/sample)  Model Size (KB)  \
0  0.594673  0.066078                    0.011277      1681.930664   
1  0.456942  0.006770                    0.040160      1455.073242   

           Feature Ranking (most to least important)  
0  ['Latitude', 'MedInc', 'Longitude', 'HouseAge'...  
1  [MedInc, Latitude, Longitude, HouseAge, AveOcc...  


**XGBoost**
---
Since California Housing contains only numeric features with no missing values, no additional preprocessing is required for XGBoost. Tree-based models are invariant to feature scaling, so the original feature values are used directly without normalization. The data is split into 60/20/20 train/validation/test sets, consistent with other models.

In [ ]:
# use original (unscaled) features for XGBoost
X_train_xgb = X_train.copy()
X_val_xgb = X_val.copy()
X_test_xgb = X_test.copy()

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:

- n_estimators: number of boosting trees in the ensemble
- max_depth: maximum depth of each decision tree
- learning_rate: step size controlling the contribution of each tree

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "subsample": 0.8,  # Enable stochasticity via row sampling
        "colsample_bytree": 0.8,  # Enable stochasticity via column sampling
        "random_state": 0,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    model.fit(X_train_xgb, y_train)

    preds = model.predict(X_val_xgb)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    return rmse


study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=50)

print("Best RMSE:", study_xgb.best_value)
print("Best params:", study_xgb.best_params)

[I 2026-04-07 21:38:06,839] A new study created in memory with name: no-name-48f105ff-200c-4e56-8c7b-d7d6a00bf93d
[I 2026-04-07 21:38:07,012] Trial 0 finished with value: 0.5213833356820741 and parameters: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03707098322903158}. Best is trial 0 with value: 0.5213833356820741.
[I 2026-04-07 21:38:07,117] Trial 1 finished with value: 0.878717521847233 and parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.0024311516573395763}. Best is trial 0 with value: 0.5213833356820741.
[I 2026-04-07 21:38:07,167] Trial 2 finished with value: 0.8278586310086403 and parameters: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.007960059431591466}. Best is trial 0 with value: 0.5213833356820741.
[I 2026-04-07 21:38:07,450] Trial 3 finished with value: 0.9027537048747342 and parameters: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.0013693845690427437}. Best is trial 0 with value: 0.5213833356820741.
[I 2026-04-07 

Best RMSE: 0.4518473094038876
Best params: {'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.08658395187983835}


**Final Evaluation**

In [ ]:
best_params_xgb = study_xgb.best_params
seeds = [1, 2, 3]

final_rmse_xgb, final_mae_xgb = [], []

for seed in seeds:
    model = XGBRegressor(
        **best_params_xgb,
        subsample=0.8,  # Enable stochasticity via row sampling
        colsample_bytree=0.8,  # Enable stochasticity via column sampling
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train_xgb, y_train)

    preds = model.predict(X_test_xgb)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    final_rmse_xgb.append(rmse)
    final_mae_xgb.append(mae)

    print(f"seed={seed}, rmse={rmse:.10f}, mae={mae:.10f}")

print(final_rmse_xgb)
print(final_mae_xgb)
print(f"XGBoost RMSE: {np.mean(final_rmse_xgb):.4f} ± {np.std(final_rmse_xgb):.4f}")
print(f"XGBoost MAE: {np.mean(final_mae_xgb):.4f} ± {np.std(final_mae_xgb):.4f}")

seed=1, rmse=0.4642331691, mae=0.2982219506
seed=2, rmse=0.4647652666, mae=0.2976825563
seed=3, rmse=0.4660730409, mae=0.2994279878
[np.float64(0.46423316912271706), np.float64(0.46476526663172957), np.float64(0.4660730409360873)]
[0.2982219506071889, 0.29768255628372814, 0.29942798781601265]
XGBoost RMSE: 0.4650 ± 0.0008
XGBoost MAE: 0.2984 ± 0.0007


**Inference Cost**

In [ ]:
start = time.time()
_ = model.predict(X_test_xgb)
elapsed = time.time() - start

print(f"Inference time per sample: {elapsed/len(X_test_xgb)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model)) / 1024:.2f} KB")

Inference time per sample: 0.0013 ms
Model size: 2929.42 KB


**Feature Importance**

In [ ]:
# get feature importance from final trained model
importance = model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": x.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df)

xgb_feature_ranking = feature_importance_df["Feature"].tolist()

      Feature  Importance
0      MedInc    0.366528
7   Longitude    0.146605
5    AveOccup    0.142654
6    Latitude    0.140731
2    AveRooms    0.079856
1    HouseAge    0.056698
3   AveBedrms    0.039745
4  Population    0.027184


**Export Results**

In [ ]:
# export obtained results, including feature importance by ranking
new_row = {
    "Model": ["XGBoost"],
    "Dataset": ["California Housing"],
    "Task": ["Regression"],
    "RMSE Mean": [np.mean(final_rmse_xgb)],
    "RMSE Std": [np.std(final_rmse_xgb)],
    "MAE Mean": [np.mean(final_mae_xgb)],
    "MAE Std": [np.std(final_mae_xgb)],
    "Inference Time (ms/sample)": [elapsed/len(X_test_xgb)*1000],
    "Model Size (KB)": [len(pickle.dumps(model)) / 1024],
    "Feature Ranking (most to least important)": [xgb_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_california_housing.csv", index=False)

print(results_df)

            Model             Dataset        Task  RMSE Mean  RMSE Std  \
0          TabNet  California Housing  Regression   0.834352  0.085951   
1  FT-Transformer  California Housing  Regression   0.677785  0.018639   
2         XGBoost  California Housing  Regression   0.465024  0.000773   

   MAE Mean   MAE Std  Inference Time (ms/sample)  Model Size (KB)  \
0  0.594673  0.066078                    0.011277      1681.930664   
1  0.456942  0.006770                    0.040160      1455.073242   
2  0.298444  0.000730                    0.001333      2929.419922   

           Feature Ranking (most to least important)  
0  ['Latitude', 'MedInc', 'Longitude', 'HouseAge'...  
1  [MedInc, Latitude, Longitude, HouseAge, AveOcc...  
2  [MedInc, Longitude, AveOccup, Latitude, AveRoo...  


**Random Forest**

Since California Housing contains only numeric features with no missing values, no additional preprocessing is required for Random Forest. As a tree-based model, similar to XGboost, Random Forest is invariant to feature scaling, so the original feature values are used directly without normalization. The data is split into 60/20/20 train/validation/test sets, consistent with other models.

In [ ]:
# use original (unscaled) features for Random Forest
X_train_rf = X_train.copy()
X_val_rf = X_val.copy()
X_test_rf = X_test.copy()

**Hyperparameter Tuning**

Optuna is used to tune the following parameters over 50 trials:

- n_estimators: number of trees in the forest
- max_depth: maximum depth of each decision tree
- max_features: number of features considered at each split

In [ ]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 0,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    model.fit(X_train_rf, y_train)

    preds = model.predict(X_val_rf)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    return rmse


study_rf = optuna.create_study(direction="minimize")
study_rf.optimize(objective_rf, n_trials=50)

print("Best RMSE:", study_rf.best_value)
print("Best params:", study_rf.best_params)

[I 2026-04-07 21:38:21,571] A new study created in memory with name: no-name-76fcb8d5-d3aa-457b-90e1-67330ed48960
[I 2026-04-07 21:38:21,915] Trial 0 finished with value: 0.7571760138353438 and parameters: {'n_estimators': 300, 'max_depth': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7571760138353438.
[I 2026-04-07 21:38:22,139] Trial 1 finished with value: 0.658092492972155 and parameters: {'n_estimators': 200, 'max_depth': 6, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.658092492972155.
[I 2026-04-07 21:38:23,061] Trial 2 finished with value: 0.5508779133154358 and parameters: {'n_estimators': 300, 'max_depth': 10, 'max_features': None}. Best is trial 2 with value: 0.5508779133154358.
[I 2026-04-07 21:38:24,188] Trial 3 finished with value: 0.5655314543289803 and parameters: {'n_estimators': 400, 'max_depth': 9, 'max_features': None}. Best is trial 2 with value: 0.5508779133154358.
[I 2026-04-07 21:38:25,125] Trial 4 finished with value: 0.6114872962846885 and

Best RMSE: 0.5213761715832484
Best params: {'n_estimators': 200, 'max_depth': 12, 'max_features': 'log2'}


**Final Evaluation**

In [ ]:
best_params_rf = study_rf.best_params
seeds = [1, 2, 3]

final_rmse_rf, final_mae_rf = [], []

for seed in seeds:
    model = RandomForestRegressor(
        **best_params_rf,
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train_rf, y_train)

    preds = model.predict(X_test_rf)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    final_rmse_rf.append(rmse)
    final_mae_rf.append(mae)

    print(f"seed={seed}, rmse={rmse:.10f}, mae={mae:.10f}")

seed=1, rmse=0.5227589009, mae=0.3489512807
seed=2, rmse=0.5253928775, mae=0.3517065478
seed=3, rmse=0.5217650697, mae=0.3487336495


**Inference Cost**

In [ ]:
start = time.time()
_ = model.predict(X_test_rf)
elapsed_rf = time.time() - start

print(f"Inference time per sample: {elapsed_rf/len(X_test_rf)*1000:.4f} ms")
print(f"Model size: {len(pickle.dumps(model)) / 1024:.2f} KB")

Inference time per sample: 0.0089 ms
Model size: 40655.29 KB


**Feature Importance**

In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": x.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_feature_ranking = feature_importance_df["Feature"].tolist()

print(feature_importance_df)

      Feature  Importance
0      MedInc    0.440374
5    AveOccup    0.125454
6    Latitude    0.119834
7   Longitude    0.118313
2    AveRooms    0.093908
1    HouseAge    0.051454
3   AveBedrms    0.029415
4  Population    0.021247


**Export Results**

In [ ]:
new_row = {
    "Model": ["Random Forest"],
    "Dataset": ["California Housing"],
    "Task": ["Regression"],
    "RMSE Mean": [np.mean(final_rmse_rf)],
    "RMSE Std": [np.std(final_rmse_rf)],
    "MAE Mean": [np.mean(final_mae_rf)],
    "MAE Std": [np.std(final_mae_rf)],
    "Inference Time (ms/sample)": [elapsed_rf/len(X_test_rf)*1000],
    "Model Size (KB)": [len(pickle.dumps(model)) / 1024],
    "Feature Ranking (most to least important)": [rf_feature_ranking]
}

results_df = pd.concat([results_df, pd.DataFrame(new_row)], ignore_index=True)
results_df.to_csv("results/results_california_housing.csv", index=False)
print(results_df)

            Model             Dataset        Task  RMSE Mean  RMSE Std  \
0          TabNet  California Housing  Regression   0.834352  0.085951   
1  FT-Transformer  California Housing  Regression   0.677785  0.018639   
2         XGBoost  California Housing  Regression   0.465024  0.000773   
3   Random Forest  California Housing  Regression   0.523306  0.001531   

   MAE Mean   MAE Std  Inference Time (ms/sample)  Model Size (KB)  \
0  0.594673  0.066078                    0.011277      1681.930664   
1  0.456942  0.006770                    0.040160      1455.073242   
2  0.298444  0.000730                    0.001333      2929.419922   
3  0.349797  0.001353                    0.008856     40655.289062   

           Feature Ranking (most to least important)  
0  ['Latitude', 'MedInc', 'Longitude', 'HouseAge'...  
1  [MedInc, Latitude, Longitude, HouseAge, AveOcc...  
2  [MedInc, Longitude, AveOccup, Latitude, AveRoo...  
3  [MedInc, AveOccup, Latitude, Longitude, AveRoo...  
